In [1]:
import numpy as np
import healpy as hp
#from scipy.interpolate import interp1d
import random
import input
from pathlib import Path

# Use the normal or smoothed tensor fields: '_V2' (0.1Mpc) or '_smooth' (0.5Mpc) or '_smooth_1Mpc' (1Mpc)
smooth_flag='_smooth'

# the number of galaxies per square arcminute to extract
gpam = 0.6

# The NSIDE value of the shear and weight maps
nside = 8192

#npix = 12*nside**2
#pix_size = np.sqrt(4.*np.pi*(180./np.pi*60)**2/npix)
#print('pixel size of shear maps is %f arcmin per side' % pix_size)

zfile=np.loadtxt("z2ts.txt",delimiter=',')
z_list=np.flip(zfile[1:58,0])
snaplist = np.flip(zfile[1:58,1].astype(int))
n_slices = np.size(z_list)

In [2]:
z_list

array([3.0361, 2.9412, 2.8506, 2.7361, 2.6545, 2.5765, 2.4775, 2.4068,
       2.3168, 2.2524, 2.1703, 2.0923, 2.018 , 1.9472, 1.8797, 1.7994,
       1.7384, 1.68  , 1.6104, 1.5443, 1.4938, 1.4334, 1.3759, 1.321 ,
       1.2584, 1.2088, 1.152 , 1.1069, 1.0552, 1.006 , 0.9591, 0.8646,
       0.824 , 0.7788, 0.7358, 0.6948, 0.6557, 0.6184, 0.5777, 0.5391,
       0.5022, 0.4714, 0.4337, 0.4017, 0.3636, 0.3347, 0.3035, 0.2705,
       0.2423, 0.2123, 0.1837, 0.1538, 0.1279, 0.1008, 0.0749, 0.0502,
       0.0245])

In [3]:
print(np.size(snaplist))
snaplist

57


array([121, 124, 127, 131, 134, 137, 141, 144, 148, 151, 155, 159, 163,
       167, 171, 176, 180, 184, 189, 194, 198, 203, 208, 213, 219, 224,
       230, 235, 241, 247, 253, 266, 272, 279, 286, 293, 300, 307, 315,
       323, 331, 338, 347, 355, 365, 373, 382, 392, 401, 411, 421, 432,
       442, 453, 464, 475, 487])

In [4]:
def fname_tidal(snap_no):
    return "/global/cfs/cdirs/lsst/groups/WL/projects/wl-massmap/IA-infusion//SkySim5000/tidalfield/tidal_field_map_"+str(snaplist[snap_no])+"_allsky"+smooth_flag+".npy"

def fname_density(snap_no):
    return "/global/cfs/cdirs/lsst/groups/WL/projects/wl-massmap/IA-infusion//SkySim5000//density/density_map_"+str(snaplist[snap_no])+"_dens_allsky.npy"
    
def hpmap_density(snap_no):
    hpmap_den = np.load(fname_density(snap_no))
    # Normalize the maps correctly:       
    mean_rho = (np.mean(hpmap_den)) # no factor of 8.0 since we have full sky data here
    hpmap_den /= mean_rho 
    hpmap_den -= 1
    return hpmap_den
    
def hpmap_tidal(snap_no):
    hpmap_tidal_ = np.load(fname_tidal(snap_no))
    Norm_empirical = 0.6525 
    hpmap_tidal_ *= Norm_empirical
    return hpmap_tidal_
    

In [5]:
snap_no = 0
print("over density:", np.min(hpmap_density(snap_no)), np.max(hpmap_density(snap_no)), np.mean(hpmap_density(snap_no)))
print("s11:",np.min(hpmap_tidal(snap_no)[:,0]), np.max(hpmap_tidal(snap_no)[:,0]), np.mean(hpmap_tidal(snap_no)[:,0]))
print("s22:",np.min(hpmap_tidal(snap_no)[:,1]), np.max(hpmap_tidal(snap_no)[:,1]), np.mean(hpmap_tidal(snap_no)[:,1]))
print("s12:",np.min(hpmap_tidal(snap_no)[:,2]), np.max(hpmap_tidal(snap_no)[:,2]), np.mean(hpmap_tidal(snap_no)[:,2]))

over density: -0.94428676 8.554813 5.5272477e-07
s11: -0.3984845 0.91512275 -2.2689812e-05
s22: -0.38369668 1.0683722 2.3182105e-05
s12: -0.7927493 0.9389109 9.7781545e-08


In [38]:
def scale_cls(c):
    ells = np.arange(len(c))
    return ells, ells*(ells+1)*c/(np.pi*2)

hp.alm2cl(hp.map2alm(baryon_map_eta_tot_0pt5))

In [39]:
def Compute_Cl(snap_no):
    kpmap_ring = np.load(fname_tidal(snap_no))
    # Cl = hp.sphtfunc.anafast(kpmap_ring, map2=None, nspec=None, lmax=1000, mmax=None, iter=1, alm=False, pol=False, use_weights=False, datapath=None, gal_cut=0, use_pixel_weights=False)
    Cl = hp.alm2cl(hp.map2alm(kpmap_ring))
    Cl *= 8.0
    
# fname = input.kappaDir+np.str("{:5.4f}".format(zs))+"kappa.npy"
# kpmap_ring = np.load(fname)
# print("loaded kappa")
# # Compute C_ell:
# Cl = hp.sphtfunc.anafast(kpmap_ring, map2=None, nspec=None, lmax=5000, mmax=None, iter=1, alm=False, pol=False, use_weights=False, datapath=None, gal_cut=0, use_pixel_weights=False)
# Cl *= 8.0
# print("Got C_ell")
# fname = input.kappaDir+np.str("{:5.4f}".format(zs))+"Cl.dat"
# np.savetxt(fname,Cl)
# print("Saved C_ell")

In [40]:
Compute_Cl(0)

TypeError: bad number of pixels

In [ ]:
# Normalize the maps correctly:       
mean_rho = (np.mean(hpmap_density)) # no factor of 8.0 since we have full sky data here
hpmap_density /= mean_rho 
hpmap_density -= 1

Norm_empirical = 0.6525 

hpmap_tidal *= Norm_empirical

print("over density:",np.min(hpmap_density), np.max(hpmap_density), np.mean(hpmap_density))
print("s11:",np.min(hpmap_tidal[:,0]), np.max(hpmap_tidal[:,0]), np.mean(hpmap_tidal[:,0]))
print("s22:",np.min(hpmap_tidal[:,1]), np.max(hpmap_tidal[:,1]), np.mean(hpmap_tidal[:,1]))
print("s12:",np.min(hpmap_tidal[:,2]), np.max(hpmap_tidal[:,2]), np.mean(hpmap_tidal[:,2]))

In [ ]:
#load mass maps and tidal tensor
        fname = "../../density/density_map_"+str(snaplist[plane])+"_dens_allsky.npy"
        #fname = input.deltaDir+"density_map_"+str(snaplist[plane])+"_dens_allsky_smooth.npy"

        print("opening ", fname) 
        hpmap_density = np.load(fname)
        print("Got density file!")

In [ ]:
fname = "/global/cfs/cdirs/lsst/groups/WL/projects/wl-massmap/IA-infusion//SkySim5000/tidalfield/tidalfield/tidal_field_map_"+str(snaplist[plane])+"_allsky"+smooth_flag+".npy"
print("opening ", fname) 
hpmap_tidal = np.load(fname)
print("Got tidal file!")


# Normalize the maps correctly:       
mean_rho = (np.mean(hpmap_density)) # no factor of 8.0 since we have full sky data here
hpmap_density /= mean_rho 
hpmap_density -= 1

Norm_empirical = 0.6525 

hpmap_tidal *= Norm_empirical

print("over density:",np.min(hpmap_density), np.max(hpmap_density), np.mean(hpmap_density))
print("s11:",np.min(hpmap_tidal[:,0]), np.max(hpmap_tidal[:,0]), np.mean(hpmap_tidal[:,0]))
print("s22:",np.min(hpmap_tidal[:,1]), np.max(hpmap_tidal[:,1]), np.mean(hpmap_tidal[:,1]))
print("s12:",np.min(hpmap_tidal[:,2]), np.max(hpmap_tidal[:,2]), np.mean(hpmap_tidal[:,2]))


In [ ]:
# Compute C_ell:
for zs in zsnap[28:]:

    print( "\nLoading convergence map and computing C_ell's: zs={:5.4f}\n".format(zs) )
    fname = input.kappaDir+np.str("{:5.4f}".format(zs))+"kappa.npy"
    kpmap_ring = np.load(fname)
    print("loaded kappa")

    # Compute C_ell:
    Cl = hp.sphtfunc.anafast(kpmap_ring, map2=None, nspec=None, lmax=5000, mmax=None, iter=1, alm=False, pol=False, use_weights=False, datapath=None, gal_cut=0, use_pixel_weights=False)
    Cl *= 8.0
    print("Got C_ell")
    fname = input.kappaDir+np.str("{:5.4f}".format(zs))+"Cl.dat"
    np.savetxt(fname,Cl)
    print("Saved C_ell")
    
